# SIDA: Social Media Image Deepfake Detection, Localization and Explanation
### CVPR 2025 Official Training & Inference Pipeline (Google Colab / Cloud GPU)

This notebook guides you through running **SIDA (SIDA-7B)** on Google Colab or any cloud GPU instance (NVIDIA T4, A100, or L4).

- **Hardware Requirement**: 1x GPU with $\ge 16$ GB VRAM (T4 16GB, L4 24GB, or A100 40GB/80GB).
- **Paper Architecture**: LLaVA-1.5 + CLIP-ViT-Large + SAM ViT-H with dual tokens `<DET>` and `<SEG>` fused via cross-attention.

## 1. Verify GPU Hardware Acceleration

In [ ]:
!nvidia-smi

## 2. Clone Repository and Install Dependencies

In [ ]:
# Clone repository if running standalone
!git clone https://github.com/hzlsaber/SIDA.git
%cd SIDA

# Install core dependencies
!pip install -q torch==2.1.2 torchvision==0.16.2 --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers==4.31.0 peft==0.4.0 deepspeed==0.12.6
!pip install -q opencv-python pycocotools scikit-learn gradio fastapi uvicorn
!pip install -q git+https://github.com/facebookresearch/segment-anything.git

## 3. Download Model Checkpoints
Download SAM ViT-H (`sam_vit_h_4b8939.pth`) and pretrained SIDA-7B / base model weights.

In [ ]:
import os
os.makedirs('ck', exist_ok=True)

# 1. Download SAM ViT-H
!wget -c -q --show-progress https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth -O ck/sam_vit_h_4b8939.pth

# 2. Download SIDA-7B checkpoint from Hugging Face
!pip install -q huggingface_hub
from huggingface_hub import snapshot_download
snapshot_download(repo_id="saberzl/SIDA-7B", local_dir="ck/SIDA-7B")
print("Checkpoints successfully downloaded!")

## 4. Run Inference on Example Images

In [ ]:
# Test inference using chat.py with GPU fp16 acceleration
!python chat.py \
  --version ./ck/SIDA-7B \
  --precision fp16 \
  --image_path ./examples/figure1.png \
  --prompt "Please answer begin with [CLS] for classification, if the image is tampered, output mask the tampered region."

## 5. Reduced Training / Fine-Tuning Pipeline
The full paper run requires **~48 hours on 2x NVIDIA A100 40GB** (10 epochs, 1000 steps/epoch = 10,000 steps).
Below is the reduced pipeline (1 epoch, 100 steps) running with DeepSpeed ZeRO-2 and LoRA ($r=8$) in ~30–45 minutes on a single GPU.

In [ ]:
!deepspeed --master_port=24999 train_SIDA.py \
  --version=./ck/SIDA-7B \
  --dataset_dir=./SID_Set \
  --vision_pretrained=./ck/sam_vit_h_4b8939.pth \
  --val_dataset=./SID_Set \
  --batch_size=2 \
  --exp_name=SIDA-7B-colab-reduced \
  --epochs=1 \
  --steps_per_epoch=100 \
  --lr=0.0001 \
  --ce_loss_weight=1.0 \
  --dice_loss_weight=0.5 \
  --bce_loss_weight=2.0 \
  --cls_loss_weight=1.0

## 6. Convert DeepSpeed ZeRO Checkpoint to Standalone Model & Merge LoRA

In [ ]:
# Consolidate ZeRO partitioned weights
!python zero_to_fp32.py ./runs/SIDA-7B-colab-reduced/ckpt_model ./runs/SIDA-7B-colab-reduced/pytorch_model.bin

# Merge LoRA weights into Hugging Face format
!python merge_lora_weights_and_save_hf_model.py \
  --version=./ck/SIDA-7B \
  --weight=./runs/SIDA-7B-colab-reduced/pytorch_model.bin \
  --save_path=./ck/SIDA-7B-finetuned

print("Fine-tuned model successfully merged and saved to ./ck/SIDA-7B-finetuned")

## 7. Hardware Training Time Extrapolation Table

| Hardware Setup | GPUs | Precision | Effective Speed / Step | Full Training (10,000 steps) | Reduced Run (100 steps) |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **2x NVIDIA A100 40GB (Paper Setup)** | 2 | BF16 / ZeRO-2 | ~17.2s / batch | **~48.0 Hours** | ~28 Minutes |
| **1x NVIDIA A100 80GB (Cloud)** | 1 | BF16 / ZeRO-2 | ~31.0s / batch | **~86.1 Hours** | ~52 Minutes |
| **1x NVIDIA L4 24GB (Colab Pro)** | 1 | FP16 / ZeRO-2 | ~58.0s / batch | **~161.1 Hours** | ~1.6 Hours |
| **1x NVIDIA T4 16GB (Colab Free)** | 1 | FP16 / GradAcc | ~105.0s / batch | **~291.6 Hours** | ~2.9 Hours |
| **Intel i5-12500H (Local CPU Fallback)** | 0 (16-thread CPU) | FP32 (No CUDA) | ~550.0s / step | **~1,527 Hours (~63 Days)** | ~15.3 Hours |

*Conclusion*: Local CPU execution is suitable for fast inference and evaluation on small subsets, while cloud GPUs (T4/A100) are required for gradient updates and backpropagation.